# Cyclistic — Preparación de datos para visualización

## Objetivo

Este notebook corresponde a la etapa de presentación de resultados del caso de estudio Cyclistic, desarrollado como trabajo final del Google Data Analytics Professional Certificate.

A partir del conjunto de datos consolidado y depurado, se preparan tablas resumidas que permiten construir las visualizaciones del análisis en Tableau.

El objetivo es transformar los resultados del análisis en conjuntos de datos más manejables, facilitando la elaboración de gráficos y la comunicación de los principales hallazgos.

## Datos utilizados

Se utiliza el archivo procesado correspondiente al período de septiembre de 2025 a agosto de 2026, conservando exclusivamente los viajes comprendidos dentro de ese intervalo.

**Entorno de trabajo:** este notebook fue desarrollado en Google Colab utilizando archivos almacenados en Google Drive. Las rutas corresponden al entorno original de trabajo y deben adaptarse para ejecutar el análisis en otro entorno.

## Procedimientos realizados

Se generan y exportan tablas resumidas para visualizar:

- La distribución de viajes entre miembros anuales y usuarios casuales.
- Las diferencias en la duración de los viajes, considerando dos escenarios: con y sin valores extremos.
- Los patrones de uso según el día de la semana.
- La distribución horaria de los viajes, diferenciando días laborables y fines de semana.
- La evolución mensual de la cantidad de viajes.
- La distribución de viajes según el tipo de bicicleta.

**Propósito:** disponer de tablas organizadas para construir una historia de datos en Tableau que permita comunicar las diferencias de comportamiento entre los dos grupos de usuarios.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import drive

In [ ]:
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# Ruta del archivo procesado
ruta = "/content/drive/MyDrive/Colab Notebooks/cyclistic/datos_procesados/cyclistic_202509_202608_limpio.csv"

# Cargar los datos
cyclistic = pd.read_csv(
    ruta,
    parse_dates=["started_at", "ended_at"]
)

# Filtrar el período del caso de estudio
cyclistic_periodo = cyclistic[
    (cyclistic["started_at"] >= "2025-09-01") &
    (cyclistic["started_at"] < "2026-09-01")
].copy()

print("Viajes cargados:", len(cyclistic_periodo))

Viajes cargados: 6115679


## Preparación de tablas: distribución y duración de viajes

En esta etapa se generan las primeras tablas resumidas que se utilizarán para construir las visualizaciones en Tableau.

### Distribución de viajes por tipo de usuario

Se agrupan los registros según el tipo de usuario (`member` o `casual`) y se calcula la cantidad de viajes y su participación porcentual sobre el total.

### Comparación de la duración de los viajes

Se calculan la media y la mediana de duración para ambos grupos, considerando dos escenarios:

- **Todos los viajes:** se incluyen los registros con valores extremos.
- **Sin valores extremos:** se excluyen del cálculo los viajes previamente identificados como extremos.

Esta comparación permite visualizar cómo influyen los valores extremos sobre las estadísticas de duración y evaluar las diferencias entre ambos grupos.

Finalmente, las tablas se exportan en formato CSV para utilizarlas como fuentes de datos en Tableau.

In [ ]:
# Crear carpeta para las tablas de Tableau
import os

carpeta_share = "/content/drive/MyDrive/Colab Notebooks/cyclistic/datos_share"
os.makedirs(carpeta_share, exist_ok=True)

# Resumir viajes por tipo de usuario
viajes_usuario = (
    cyclistic_periodo
    .groupby("member_casual")
    .size()
    .reset_index(name="viajes")
)

# Calcular participación porcentual
viajes_usuario["porcentaje"] = (
    viajes_usuario["viajes"] /
    viajes_usuario["viajes"].sum() * 100
).round(2)

# Exportar a CSV
viajes_usuario.to_csv(
    f"{carpeta_share}/viajes_por_usuario.csv",
    index=False
)

display(viajes_usuario)

,member_casual,viajes,porcentaje
0,casual,2160125,35.32
1,member,3955554,64.68


In [ ]:
# Duración considerando todos los viajes
duracion_todos = (
    cyclistic_periodo
    .groupby("member_casual")["ride_length_min"]
    .agg(viajes="count", media="mean", mediana="median")
    .reset_index()
)

duracion_todos["escenario"] = "Todos los viajes"

# Duración excluyendo valores extremos
duracion_sin_extremos = (
    cyclistic_periodo[cyclistic_periodo["extremo"] == 0]
    .groupby("member_casual")["ride_length_min"]
    .agg(viajes="count", media="mean", mediana="median")
    .reset_index()
)

duracion_sin_extremos["escenario"] = "Sin extremos"

# Unificar resultados
duracion_share = pd.concat(
    [duracion_todos, duracion_sin_extremos],
    ignore_index=True
)

duracion_share[["media", "mediana"]] = (
    duracion_share[["media", "mediana"]].round(2)
)

# Exportar
duracion_share.to_csv(
    f"{carpeta_share}/duracion_por_usuario.csv",
    index=False
)

display(duracion_share)

,member_casual,viajes,media,mediana,escenario
0,casual,2160125,20.66,10.86,Todos los viajes
1,member,3955554,12.37,8.55,Todos los viajes
2,casual,2063402,16.42,11.15,Sin extremos
3,member,3863438,11.63,8.71,Sin extremos


In [ ]:
import os

print(os.listdir(carpeta_share))

['viajes_por_usuario.csv', 'duracion_por_usuario.csv']


## Preparación de tablas: patrones de uso y tipos de bicicleta

En esta etapa se generan las tablas que permiten visualizar las diferencias en los patrones de uso entre miembros anuales y usuarios casuales.

### Distribución semanal y horaria

Se agrupan los viajes según el día de la semana y la hora de inicio, diferenciando entre ambos tipos de usuario.

Además, se distingue entre días laborables y fines de semana, utilizando promedios diarios para facilitar las comparaciones.

### Evolución mensual

Se calcula la cantidad de viajes por mes y tipo de usuario, con el objetivo de visualizar los cambios en la demanda a lo largo del período estudiado y comparar los patrones estacionales.

### Tipos de bicicleta

Se agrupan los viajes según el tipo de bicicleta utilizado y la categoría de usuario, calculando las cantidades y proporciones correspondientes.

Estas tablas permiten comparar las preferencias de ambos grupos y complementar los resultados obtenidos en los análisis anteriores.

Finalmente, los resultados se exportan en formato CSV para construir las visualizaciones en Tableau.

In [ ]:
# Crear día de la semana (lunes = 0)
cyclistic_periodo["dia_semana"] = (
    cyclistic_periodo["started_at"].dt.dayofweek
)

# Contar viajes por usuario y día
viajes_dia = (
    cyclistic_periodo
    .groupby(["member_casual", "dia_semana"])
    .size()
    .reset_index(name="viajes")
)

# Porcentaje dentro de cada tipo de usuario
viajes_dia["porcentaje"] = (
    viajes_dia["viajes"] /
    viajes_dia.groupby("member_casual")["viajes"].transform("sum")
    * 100
).round(2)

# Agregar nombres de días
dias = {
    0: "Lunes",
    1: "Martes",
    2: "Miércoles",
    3: "Jueves",
    4: "Viernes",
    5: "Sábado",
    6: "Domingo"
}

viajes_dia["nombre_dia"] = viajes_dia["dia_semana"].map(dias)

# Exportar
viajes_dia.to_csv(
    f"{carpeta_share}/viajes_por_dia.csv",
    index=False
)

display(viajes_dia)

,member_casual,dia_semana,viajes,porcentaje,nombre_dia
0,casual,0,257881,11.94,Lunes
1,casual,1,246857,11.43,Martes
2,casual,2,247902,11.48,Miércoles
3,casual,3,268025,12.41,Jueves
4,casual,4,334557,15.49,Viernes
5,casual,5,456662,21.14,Sábado
6,casual,6,348241,16.12,Domingo
7,member,0,566150,14.31,Lunes
8,member,1,630934,15.95,Martes
9,member,2,631953,15.98,Miércoles


In [ ]:
# Identificar hora y tipo de día
cyclistic_periodo["hora_inicio"] = (
    cyclistic_periodo["started_at"].dt.hour
)

cyclistic_periodo["tipo_dia"] = (
    cyclistic_periodo["started_at"]
    .dt.dayofweek
    .map(lambda d: "Fin de semana" if d >= 5 else "Laborable")
)

# Contar viajes por grupo, tipo de día y hora
viajes_hora = (
    cyclistic_periodo
    .groupby(["member_casual", "tipo_dia", "hora_inicio"])
    .size()
    .reset_index(name="viajes")
)

# Cantidad de días de cada tipo en el período
calendario = pd.date_range(
    start="2025-09-01",
    end="2026-08-31",
    freq="D"
)

dias_laborables = (calendario.dayofweek < 5).sum()
dias_fin_semana = (calendario.dayofweek >= 5).sum()

cantidad_dias = {
    "Laborable": dias_laborables,
    "Fin de semana": dias_fin_semana
}

# Promedio de viajes por día de cada tipo
viajes_hora["cantidad_dias"] = (
    viajes_hora["tipo_dia"].map(cantidad_dias)
)

viajes_hora["promedio_diario"] = (
    viajes_hora["viajes"] / viajes_hora["cantidad_dias"]
).round(2)

# Exportar para Tableau
viajes_hora.to_csv(
    f"{carpeta_share}/viajes_por_hora_tipo_dia.csv",
    index=False
)

display(viajes_hora.head(10))

print("Días laborables:", dias_laborables)
print("Días de fin de semana:", dias_fin_semana)
print("Filas exportadas:", len(viajes_hora))

,member_casual,tipo_dia,hora_inicio,viajes,cantidad_dias,promedio_diario
0,casual,Fin de semana,0,26235,104,252.26
1,casual,Fin de semana,1,18676,104,179.58
2,casual,Fin de semana,2,12127,104,116.61
3,casual,Fin de semana,3,6676,104,64.19
4,casual,Fin de semana,4,4084,104,39.27
5,casual,Fin de semana,5,3645,104,35.05
6,casual,Fin de semana,6,5873,104,56.47
7,casual,Fin de semana,7,9532,104,91.65
8,casual,Fin de semana,8,17172,104,165.12
9,casual,Fin de semana,9,28618,104,275.17


Días laborables: 261
Días de fin de semana: 104
Filas exportadas: 96


In [ ]:
viajes_mes = (
    cyclistic_periodo
    .groupby([
        cyclistic_periodo["started_at"].dt.to_period("M"),
        "member_casual"
    ])
    .size()
    .reset_index(name="viajes")
)

# Convertir el mes a una fecha reconocible por Tableau
viajes_mes["started_at"] = (
    viajes_mes["started_at"]
    .dt.to_timestamp()
)

viajes_mes = viajes_mes.rename(
    columns={"started_at": "mes"}
)

# Exportar
viajes_mes.to_csv(
    f"{carpeta_share}/viajes_por_mes.csv",
    index=False
)

display(viajes_mes)
print("Filas:", len(viajes_mes))

,mes,member_casual,viajes
0,2025-09-01,casual,265268
1,2025-09-01,member,449294
2,2025-10-01,casual,224038
3,2025-10-01,member,422058
4,2025-11-01,casual,99091
5,2025-11-01,member,257421
6,2025-12-01,casual,28090
7,2025-12-01,member,112455
8,2026-01-01,casual,24740
9,2026-01-01,member,113042


Filas: 24


In [ ]:
# Contar viajes por tipo de usuario y bicicleta
viajes_bicicleta = (
    cyclistic_periodo
    .groupby(["member_casual", "rideable_type"])
    .size()
    .reset_index(name="viajes")
)

# Calcular porcentajes dentro de cada grupo
viajes_bicicleta["porcentaje"] = (
    viajes_bicicleta["viajes"] /
    viajes_bicicleta.groupby("member_casual")["viajes"].transform("sum")
    * 100
).round(2)

# Exportar para Tableau
viajes_bicicleta.to_csv(
    f"{carpeta_share}/viajes_por_bicicleta.csv",
    index=False
)

display(viajes_bicicleta)

print("Filas exportadas:", len(viajes_bicicleta))

,member_casual,rideable_type,viajes,porcentaje
0,casual,classic_bike,570174,26.40
1,casual,electric_bike,1589951,73.60
2,member,classic_bike,1263510,31.94
3,member,electric_bike,2692044,68.06


Filas exportadas: 4


## Visualización de resultados

Las tablas generadas en este notebook se utilizaron para construir una historia interactiva en Tableau Public.

La visualización presenta los principales hallazgos del análisis, compara los comportamientos de los miembros anuales y los usuarios casuales y desarrolla las conclusiones del caso de estudio.

**Visualización:** [Cyclistic — Análisis de usuarios (Tableau Public)](https://public.tableau.com/app/profile/cesar.canessa/viz/Cyclistic-Analisisdeusuarios/Cyclistic-Historia)